# L3: Prompt Engineering for Video Generation

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [ ]:
import time
from IPython.display import Image, Markdown, display, Video
from google import genai
from google.genai import types

In [ ]:
import os
from helper import authenticate

credentials, project_id = authenticate()
client = genai.Client(
    project=project_id,
    location="global",
    credentials=credentials,
    http_options=types.HttpOptions(
         base_url=os.getenv("GOOGLE_VERTEX_BASE_URL")
    )
)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different run results:</b> The output generated by AI models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

In [ ]:
IMAGE_MODEL_ID = "gemini-3.1-flash-image-preview"
TEXT_MODEL_ID = "gemini-3-flash-preview"
VIDEO_MODEL_ID = "veo-3.1-fast-generate-001"

In [ ]:
def show_video(video, filename):
    with open(filename, "wb") as out_file:
        out_file.write(video)
    display(Video(filename, embed=True, width=600))

## Simple prompt

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Please wait:</b> This notebook cell may take a few minutes to run.</p>

In [ ]:
operation = client.models.generate_videos(
    model=VIDEO_MODEL_ID,
    prompt="A person explaining the Pythagorean theorem.",
    config=types.GenerateVideosConfig(
        aspect_ratio="16:9",
        number_of_videos=1,
        duration_seconds=8,
        resolution="720p",
        person_generation="allow_adult",
        generate_audio=True,
    ),
)

while not operation.done:
    time.sleep(15)
    operation = client.operations.get(operation)

if operation.response:
    show_video(operation.result.generated_videos[0].video.video_bytes, 
               "original.mp4")

## Generate input image

In [ ]:
display(Image("slide-image.png", width=500))

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Please wait:</b> This notebook cell may take a minute to run.</p>

In [ ]:
with open("slide-image.png", "rb") as f:
    image = f.read()

response = client.models.generate_content(
    model=IMAGE_MODEL_ID,
    contents=[
        types.Part.from_bytes(
            data=image,
            mime_type="image/png",
        ),
        """Generate a professor in a classroom full of students in
        front of the provided slide image.""",
    ],
    config=types.GenerateContentConfig(
        response_modalities=['IMAGE'],
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
        ),
    ),
)

for part in response.candidates[0].content.parts:
    if part.inline_data:
        image_data=part.inline_data.data
        display(Image(data=image_data, width=500))

with open("video-image.png", "wb") as image_file:
    image_file.write(image_data)

## Prompt enhancement

In [ ]:
subject = "a professor"
action = "giving a lecture on the Pythagorean theorem"
scene = "in a classroom, presenting in front of students"
style = "photorealistic"

camera_angle = "eye-level shot"
camera_movement = "zoom in"

sound_effects = "clicking of computer keys"
dialogue = """
When you square a side length, like a or b, you are literally calculating 
the area of a square built perfectly off that side.
"""

In [ ]:
keywords = [subject, action, scene, style, camera_angle, 
            camera_movement, sound_effects, dialogue]

gemini_prompt = f"""
Your task is to expand the following keywords into a single, high-fidelity, 
descriptive prompt for video generation. Every single keyword MUST be 
included. Synthesize them into a single, cohesive, and cinematic 
instruction. Do not add any new core concepts. Output ONLY the final 
prompt string, without any introduction or explanation. Mandatory Keywords: 
{",".join(keywords)}
"""

response = client.models.generate_content(
    model=TEXT_MODEL_ID,
    contents=gemini_prompt,
)

video_prompt = response.text
display(Markdown(response.text))

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Please wait:</b> This notebook cell may take a few minutes to run.</p>

In [ ]:
operation = client.models.generate_videos(
    model=VIDEO_MODEL_ID,
    prompt=video_prompt,
    image=types.Image.from_file(location="video-image.png"),
    config=types.GenerateVideosConfig(
        aspect_ratio="16:9",
        number_of_videos=1,
        duration_seconds=8,
        resolution="720p",
        person_generation="allow_adult",
        generate_audio=True,
    ),
)

while not operation.done:
    time.sleep(15)
    operation = client.operations.get(operation)

if operation.response:
    show_video(operation.result.generated_videos[0].video.video_bytes, 
               "enhanced.mp4")

**Resources**
- Model [documentation](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/veo/3-1-generate#3.1-fast-generate-001)
- Video [prompting guide](https://cloud.google.com/blog/products/ai-machine-learning/ultimate-prompting-guide-for-veo-3-1)